In [0]:
%run /Workspace/Users/ashutoshp2@icloud.com/ecommerce_data_pipeline/resources/utils

In [0]:
logger = get_logger()

In [0]:
ingestion_configs = parse_config_from_json("/Workspace/Users/ashutoshp2@icloud.com/ecommerce_data_pipeline/config/ingestion_configs.json")

In [0]:
def ingest(module: str):
    try:
        configs = get_value_from_dict(ingestion_configs, module)

        schemas = configs["schema_alias"]
        src_path = configs["file_path"]
        target_table = configs["target_table"]
        src_format = configs["file_format"]

        if src_format == "json":
            df = json_reader(src_path)
        elif src_format == "csv":
            df = csv_reader(src_path)
        elif src_format == "excel":
            sheet_name = configs["params"]["sheet_name"]
            df = excel_reader(src_path, sheet_name)
        else:
            raise Exception("Invalid file format")
        
        # Apply schema alias#
        df = apply_schema_alias(df, schemas)
        # write bronze tables #
        delta_writer(df, target_table)

    except Exception as e:
        logger.error(f"Error ingesting {module}: {e}")

In [0]:
def main_ingestion():
    ingestion_candidates = list(ingestion_configs.keys())
    for module in ingestion_candidates:
        try:
            ingest(module)
        except Exception as e:
            logger.error(f"Error in main while ingesting {module}: {e}")
